# 02, kappa on everything

The first agreement between filer and NVD severity was a kappa over every CVE with either score, with a missing score counted as its own band. Most of those CVEs have only one of the two scores, so the kappa mostly measured who scored, not whether the scores agree. The registered Q4 restricts to the CVEs both scored.

In [1]:
import duckdb
import polars as pl

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_width_chars(110)
con = duckdb.connect("data/lake/unscheduled.duckdb", read_only=True)
con.execute("set TimeZone = 'UTC'").fetchall()
cut = con.execute("select data_cut from config.lake_build").fetchall()[0][0]
print("lake cut at", cut)

lake cut at 2026-11-13 06:00:00+00:00


In [2]:
from unscheduled_analysis.stats import cohen_kappa

bands = ["None", "Low", "Medium", "High", "Critical"]
code = {b.upper(): i for i, b in enumerate(bands)}
frame = con.sql('''
    select upper(coalesce(filer_cvss31_band, 'NONE')) as filer,
           upper(coalesce(nvd_cvss31_band, 'NONE')) as nvd
    from gold.cve_current
    where state <> 'REJECTED' and (filer_cvss31_band is not null or nvd_cvss31_band is not null)
''').pl()
f = [code.get(x, 0) for x in frame["filer"]]
n = [code.get(x, 0) for x in frame["nvd"]]
print(f"{frame.height:,} CVEs with either score")
print("weighted kappa on everything:", round(cohen_kappa(f, n, k=5, weights="quadratic"), 3))

249,646 CVEs with either score
weighted kappa on everything: -0.494


In [3]:
both = frame.filter((pl.col("filer") != "NONE") & (pl.col("nvd") != "NONE"))
f2 = [code[x] for x in both["filer"]]
n2 = [code[x] for x in both["nvd"]]
print(f"{both.height:,} CVEs both scored")
print("weighted kappa where both scored:", round(cohen_kappa(f2, n2, k=5, weights="quadratic"), 3))
print("one side missing:", f"{(frame.height - both.height) / frame.height:.1%} of the first population")

61,394 CVEs both scored
weighted kappa where both scored: 0.461
one side missing: 75.4% of the first population


A missing score was coded as the band None, the same code as a score of 0.0, so the first kappa was mostly a measure of which CVEs had two scores. The registered Q4 also leaves out the test window, which is held for Phase 2, so its weighted kappa on the band (report/backlog.md) is computed on fewer CVEs than the both scored row above.

What changed in the build: Q4 (packages/analysis questions/q4.py and gold.mart_filer_vs_nvd) takes the CVEs scored with CVSS 3.1 by both the filer and the NVD, reports a kappa per metric and a weighted kappa on the band, each with a cluster bootstrap interval by filer.